In [4]:
import pandas as pd
import numpy as np
import sys
!{sys.executable} -m pip install plotly
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import sys
!{sys.executable} -m pip install geopandas
import geopandas as gpd
import urllib.request
import json
import warnings
import sys
!{sys.executable} -m pip install geopy
from geopy.distance import geodesic

# ปิดการแจ้งเตือนจุกจิก
warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: โหลดข้อมูลที่คลีนแล้ว
# ==========================================
try:
    df_cleaned = pd.read_csv('../data/NY_House_Cleaned.csv')
    print("1. โหลดไฟล์สำเร็จ")
except:
      print("❌ ไม่พบไฟล์: ../data/NY_House_Cleaned.csv")

# ==========================================
# STEP 2: คำนวณ (เปลี่ยนเป็นระบบ Multi-Hubs)
# ==========================================
print("🤖 2. กำลังคำนวณดัชนีความคุ้มค่า (ระบบ Multi-Hubs)...")
df_cleaned['PRICE_PER_SQFT'] = df_cleaned['PRICE'] / df_cleaned['PROPERTYSQFT']

ECONOMIC_HUBS = [
    # --- แมนฮัตตัน (Manhattan) ---
    (40.7580, -73.9855), # 1. Midtown Manhattan (ศูนย์กลางธุรกิจหลัก)
    (40.7081, -74.0093), # 2. Financial District (Wall Street / Lower Manhattan)

    # --- บรุกลิน (Brooklyn) ---
    (40.6925, -73.9868), # 3. Downtown Brooklyn (ย่านธุรกิจใหญ่อันดับ 3 ของเมือง)

    # --- ควีนส์ (Queens) ---
    (40.7447, -73.9485), # 4. Long Island City (ศูนย์กลางเทคโนโลยี/ออฟฟิศใหม่)
    (40.7654, -73.8282), # 5. Flushing (ศูนย์กลางเศรษฐกิจชาวเอเชียที่เงินสะพัดมาก)
    (40.7024, -73.7966), # 6. Jamaica (ศูนย์กลางการคมนาคมและพาณิชย์หลักของควีนส์)

    # --- เดอะบรองซ์ (The Bronx) ---
    (40.8162, -73.9165), # 7. The Hub / South Bronx (ศูนย์กลางการค้าและช้อปปิ้งที่ใหญ่ที่สุดของเขต)

    # --- สแตเทนไอส์แลนด์ (Staten Island) ---
    (40.6437, -74.0759)  # 8. St. George (ท่าเรือเฟอร์รี่ / ศูนย์ราชการ / ศูนย์กลางเศรษฐกิจของเกาะ)
]

def min_distance_to_hubs(lat, lon):
    distances = [geodesic((lat, lon), (hub_lat, hub_lon)).miles for hub_lat, hub_lon in ECONOMIC_HUBS]
    return min(distances)

df_cleaned['DISTANCE_TO_CENTER'] = df_cleaned.apply(lambda row: min_distance_to_hubs(row['LATITUDE'], row['LONGITUDE']), axis=1)

df_cleaned['PRICE_RANK'] = df_cleaned['PRICE_PER_SQFT'].rank(pct=True)
df_cleaned['DISTANCE_RANK'] = df_cleaned['DISTANCE_TO_CENTER'].rank(pct=True)

df_cleaned['VALUE_INDEX'] = (df_cleaned['PRICE_RANK'] * 3.0) + df_cleaned['DISTANCE_RANK']

features = ['PRICE_RANK', 'DISTANCE_RANK']
X = df_cleaned[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cleaned['CLUSTER'] = kmeans.fit_predict(X_scaled)

gdf_houses = gpd.GeoDataFrame(
    df_cleaned,
    geometry=gpd.points_from_xy(df_cleaned.LONGITUDE, df_cleaned.LATITUDE),
    crs="EPSG:4326"
)

# ==========================================
# STEP 3: โหลดแผนที่ (ระบบป้องกัน Error 100%)
# ==========================================
print("🗺️ 3. กำลังดึงกรอบแผนที่...")

nyc_geojson = None
name_col = ""

try:
    url = "https://raw.githubusercontent.com/veltman/snd3/master/data/nyc-neighborhoods.geo.json"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=20) as response:
        nyc_geojson = json.loads(response.read().decode('utf-8'))

    gdf_map = gpd.GeoDataFrame.from_features(nyc_geojson["features"])
    gdf_map.set_crs(epsg=4326, inplace=True)
    name_col = 'name'
    print("โหลดแผนที่ระดับย่าน (Neighborhoods) สำเร็จ!")

except Exception as e:
    print(f"โหลดแผนที่จากเว็บไม่ได้ (Error: 404 หรือ Timeout)")
    print("ระบบกำลังสลับไปใช้แผนที่ระดับเขต (Borough) ที่ฝังรูปแบบออฟไลน์ใน Python แทน (รันผ่านแน่นอน!)...")

    path = gpd.datasets.get_path('nybb')
    gdf_map = gpd.read_file(path)
    gdf_map = gdf_map.to_crs(epsg=4326)

    nyc_geojson = json.loads(gdf_map.to_json())
    name_col = 'BoroName'

feature_key = f"properties.{name_col}"

# ==========================================
# STEP 4: ทาบพิกัด และสร้าง Map (ดึงชื่อเขต Borough มาด้วย)
# ==========================================
print("📊 4. กำลังสรุปข้อมูลและดึงชื่อเขต (Borough)...")
joined = gpd.sjoin(gdf_houses, gdf_map, how="inner", predicate="intersects")

possible_boro_cols = [col for col in joined.columns if col.lower() in ['boro_name', 'borough', 'boroname', 'county', 'sublocality']]
boro_col = possible_boro_cols[0] if possible_boro_cols else None

agg_funcs = {
    'VALUE_INDEX': 'mean',
    'PRICE_PER_SQFT': 'mean',
    'DISTANCE_TO_CENTER': 'mean',
    'PRICE': 'mean',
    'CLUSTER': lambda x: pd.Series.mode(x).iloc[0]
}
if boro_col:
    agg_funcs[boro_col] = lambda x: pd.Series.mode(x).iloc[0]

zone_stats = joined.groupby(name_col).agg(agg_funcs).reset_index()

if boro_col:
    zone_stats = zone_stats.rename(columns={boro_col: 'Borough'})
    def clean_boro(name):
        name = str(name).lower()
        if 'kings' in name or 'brooklyn' in name: return 'Brooklyn'
        if 'new york' in name or 'manhattan' in name: return 'Manhattan'
        if 'richmond' in name or 'staten' in name: return 'Staten Island'
        if 'queens' in name: return 'Queens'
        if 'bronx' in name: return 'Bronx'
        return name.title()
    zone_stats['Borough'] = zone_stats['Borough'].apply(clean_boro)
else:
    zone_stats['Borough'] = 'NYC'

cluster_means = zone_stats.groupby('CLUSTER')['VALUE_INDEX'].mean().sort_values()
cluster_labels = {
    cluster_means.index[0]: '1. The Gold Mines (Best Value)',
    cluster_means.index[1]: '2. Smart Suburbs (Good Value)',
    cluster_means.index[2]: '3. Premium Zones (Pricey)',
    cluster_means.index[3]: '4. Deal Breakers (Overpriced/Too Far)'
}
zone_stats['CLUSTER_NAME'] = zone_stats['CLUSTER'].map(cluster_labels)

zone_stats['RANK_IN_CLUSTER'] = zone_stats.groupby('CLUSTER_NAME')['VALUE_INDEX'].rank(method='min').astype(int)
zone_stats['TOTAL_IN_CLUSTER'] = zone_stats.groupby('CLUSTER_NAME')['CLUSTER_NAME'].transform('count')
zone_stats['RANK_TEXT'] = "อันดับ " + zone_stats['RANK_IN_CLUSTER'].astype(str) + " (จาก " + zone_stats['TOTAL_IN_CLUSTER'].astype(str) + " ย่าน)"

vmin = zone_stats['VALUE_INDEX'].quantile(0.05)
vmax = zone_stats['VALUE_INDEX'].quantile(0.95)

fig = px.choropleth_mapbox(
    zone_stats,
    geojson=nyc_geojson,
    locations=name_col,
    featureidkey=feature_key,
    color='VALUE_INDEX',
    color_continuous_scale="RdYlGn_r",
    range_color=[vmin, vmax],
    mapbox_style="carto-positron",
    zoom=9.5,
    center={"lat": 40.7128, "lon": -74.0060},
    opacity=0.75,
    hover_name=name_col,
    hover_data={
        'Borough': True,
        'CLUSTER_NAME': True,
        'RANK_TEXT': True,
        'PRICE': ':,.0f',
        'PRICE_PER_SQFT': ':,.0f',
        'DISTANCE_TO_CENTER': ':.3f',
        'VALUE_INDEX': ':.2f',
        name_col: False
    },
    title='💎 The Gold Mine Map: แผนที่ความคุ้มค่า (ไล่เฉดสี) พร้อมข้อมูลจัดกลุ่มจาก AI',
    width=1000, height=800
)

fig.update_traces(marker_line_color='white', marker_line_width=0.5)
fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0}, coloraxis_colorbar=dict(title="ดัชนีความคุ้มค่า<br>(เขียวเข้ม=ดีสุด)"))
fig.show(renderer="browser")
# ==========================================
# STEP 5: ตารางสรุป "ตัวตึง" ประจำแก๊งเหมืองทอง (โชว์คอลัมน์ Borough)
# ==========================================
print("\n🏆 Top 10 'The Gold Mines' (ย่านที่เจ๋งที่สุด 10 อันดับแรกในกลุ่มสีเขียว)")

gold_mine_cluster = zone_stats[zone_stats['CLUSTER_NAME'] == '1. The Gold Mines (Best Value)']
top_gold_mines = gold_mine_cluster.sort_values(by='VALUE_INDEX').head(10)

top_gold_mines = top_gold_mines.rename(columns={
    name_col: 'Neighborhood',
    'Borough': 'Borough (เขต)',
    'RANK_TEXT': 'Rank in Cluster',
    'VALUE_INDEX': 'Value Index (Lower is better)',
    'PRICE_PER_SQFT': 'Price/Sqft ($)',
    'DISTANCE_TO_CENTER': 'Distance to Hub'
})

display(top_gold_mines[['Neighborhood', 'Borough (เขต)', 'Rank in Cluster', 'Value Index (Lower is better)', 'Price/Sqft ($)', 'Distance to Hub']])


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


1. โหลดไฟล์สำเร็จ
🤖 2. กำลังคำนวณดัชนีความคุ้มค่า (ระบบ Multi-Hubs)...
🗺️ 3. กำลังดึงกรอบแผนที่...
โหลดแผนที่ระดับย่าน (Neighborhoods) สำเร็จ!
📊 4. กำลังสรุปข้อมูลและดึงชื่อเขต (Borough)...

🏆 Top 10 'The Gold Mines' (ย่านที่เจ๋งที่สุด 10 อันดับแรกในกลุ่มสีเขียว)


,Neighborhood,Borough (เขต),Rank in Cluster,Value Index (Lower is better),Price/Sqft ($),Distance to Hub
172,West Concourse,Bronx,อันดับ 1 (จาก 43 ย่าน),0.107484,77.808529,0.831428
103,Melrose South-Mott Haven North,Bronx,อันดับ 2 (จาก 43 ย่าน),0.481976,240.847478,0.408235
79,Highbridge,Bronx,อันดับ 3 (จาก 43 ย่าน),0.565758,188.472461,1.533482
86,Jamaica,Queens,อันดับ 4 (จาก 43 ย่าน),0.708584,673.084881,0.593843
84,Hunts Point,Bronx,อันดับ 5 (จาก 43 ย่าน),0.771222,263.300246,1.504510
108,Morrisania-Melrose,Bronx,อันดับ 6 (จาก 43 ย่าน),0.812889,294.710763,1.009312
89,Kew Gardens,Queens,อันดับ 7 (จาก 43 ย่าน),0.826807,259.015274,1.840698
18,Briarwood-Jamaica Hills,Queens,อันดับ 8 (จาก 43 ย่าน),0.846956,297.884332,0.906382
96,Longwood,Bronx,อันดับ 9 (จาก 43 ย่าน),0.858800,328.469973,1.000036
45,East Concourse-Concourse Village,Bronx,อันดับ 10 (จาก 43 ย่าน),0.928460,337.096766,0.902090


In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import geopandas as gpd
import urllib.request
import json
import warnings
from geopy.distance import geodesic

warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: โหลดข้อมูลที่คลีนแล้ว
# ==========================================
try:
    df_cleaned = pd.read_csv('../data/NY_House_Cleaned.csv')
    print("✅ 1. โหลดไฟล์สำเร็จ")
except:
          print("❌ ไม่พบไฟล์: ../data/NY_House_Cleaned.csv")


# ==========================================
# STEP 2: คำนวณ (เปลี่ยนเป็นระบบ Multi-Hubs)
# ==========================================
print("2. กำลังคำนวณดัชนีความคุ้มค่า (ระบบ Multi-Hubs)...")
df_cleaned['PRICE_PER_SQFT'] = df_cleaned['PRICE'] / df_cleaned['PROPERTYSQFT']

ECONOMIC_HUBS = [
    # --- แมนฮัตตัน (Manhattan) ---
    (40.7580, -73.9855), # 1. Midtown Manhattan (ศูนย์กลางธุรกิจหลัก)
    (40.7081, -74.0093), # 2. Financial District (Wall Street / Lower Manhattan)

    # --- บรุกลิน (Brooklyn) ---
    (40.6925, -73.9868), # 3. Downtown Brooklyn (ย่านธุรกิจใหญ่อันดับ 3 ของเมือง)

    # --- ควีนส์ (Queens) ---
    (40.7447, -73.9485), # 4. Long Island City (ศูนย์กลางเทคโนโลยี/ออฟฟิศใหม่)
    (40.7654, -73.8282), # 5. Flushing (ศูนย์กลางเศรษฐกิจชาวเอเชียที่เงินสะพัดมาก)
    (40.7024, -73.7966), # 6. Jamaica (ศูนย์กลางการคมนาคมและพาณิชย์หลักของควีนส์)

    # --- เดอะบรองซ์ (The Bronx) ---
    (40.8162, -73.9165), # 7. The Hub / South Bronx (ศูนย์กลางการค้าและช้อปปิ้งที่ใหญ่ที่สุดของเขต)

    # --- สแตเทนไอส์แลนด์ (Staten Island) ---
    (40.6437, -74.0759)  # 8. St. George (ท่าเรือเฟอร์รี่ / ศูนย์ราชการ / ศูนย์กลางเศรษฐกิจของเกาะ)
]

def min_distance_to_hubs(lat, lon):
    distances = [geodesic((lat, lon), (hub_lat, hub_lon)).miles for hub_lat, hub_lon in ECONOMIC_HUBS]
    return min(distances)

df_cleaned['DISTANCE_TO_CENTER'] = df_cleaned.apply(lambda row: min_distance_to_hubs(row['LATITUDE'], row['LONGITUDE']), axis=1)

df_cleaned['PRICE_RANK'] = df_cleaned['PRICE_PER_SQFT'].rank(pct=True)
df_cleaned['DISTANCE_RANK'] = df_cleaned['DISTANCE_TO_CENTER'].rank(pct=True)

df_cleaned['VALUE_INDEX'] = (df_cleaned['PRICE_RANK'] * 3.0) + df_cleaned['DISTANCE_RANK']

features = ['PRICE_RANK', 'DISTANCE_RANK']
X = df_cleaned[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cleaned['CLUSTER'] = kmeans.fit_predict(X_scaled)

gdf_houses = gpd.GeoDataFrame(
    df_cleaned,
    geometry=gpd.points_from_xy(df_cleaned.LONGITUDE, df_cleaned.LATITUDE),
    crs="EPSG:4326"
)

# ==========================================
# STEP 3: โหลดแผนที่ (ระบบป้องกัน Error 100%)
# ==========================================
print("3. กำลังดึงกรอบแผนที่...")

nyc_geojson = None
name_col = ""

try:
    url = "https://raw.githubusercontent.com/veltman/snd3/master/data/nyc-neighborhoods.geo.json"
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=20) as response:
        nyc_geojson = json.loads(response.read().decode('utf-8'))

    gdf_map = gpd.GeoDataFrame.from_features(nyc_geojson["features"])
    gdf_map.set_crs(epsg=4326, inplace=True)
    name_col = 'name'
    print("✅ โหลดแผนที่ระดับย่าน (Neighborhoods) สำเร็จ!")

except Exception as e:
    print(f"⚠️ โหลดแผนที่จากเว็บไม่ได้ (Error: 404 หรือ Timeout)")
    print("🔄 ระบบกำลังสลับไปใช้แผนที่ระดับเขต (Borough) ที่ฝังรูปแบบออฟไลน์ใน Python แทน (รันผ่านแน่นอน!)...")

    path = gpd.datasets.get_path('nybb')
    gdf_map = gpd.read_file(path)
    gdf_map = gdf_map.to_crs(epsg=4326)

    nyc_geojson = json.loads(gdf_map.to_json())
    name_col = 'BoroName'

feature_key = f"properties.{name_col}"

# ==========================================
# STEP 4: ทาบพิกัด และสร้าง Map (ดึงชื่อเขต Borough + พล็อตจุด Hubs)
# ==========================================
import plotly.graph_objects as go

print("📊 4. กำลังสรุปข้อมูล ดึงชื่อเขต และวาดแผนที่...")
joined = gpd.sjoin(gdf_houses, gdf_map, how="inner", predicate="intersects")

possible_boro_cols = [col for col in joined.columns if col.lower() in ['boro_name', 'borough', 'boroname', 'county', 'sublocality']]
boro_col = possible_boro_cols[0] if possible_boro_cols else None

agg_funcs = {
    'VALUE_INDEX': 'mean',
    'PRICE_PER_SQFT': 'mean',
    'DISTANCE_TO_CENTER': 'mean',
    'PRICE': 'mean',
    'CLUSTER': lambda x: pd.Series.mode(x).iloc[0]
}
if boro_col:
    agg_funcs[boro_col] = lambda x: pd.Series.mode(x).iloc[0]

zone_stats = joined.groupby(name_col).agg(agg_funcs).reset_index()

if boro_col:
    zone_stats = zone_stats.rename(columns={boro_col: 'Borough'})
    def clean_boro(name):
        name = str(name).lower()
        if 'kings' in name or 'brooklyn' in name: return 'Brooklyn'
        if 'new york' in name or 'manhattan' in name: return 'Manhattan'
        if 'richmond' in name or 'staten' in name: return 'Staten Island'
        if 'queens' in name: return 'Queens'
        if 'bronx' in name: return 'Bronx'
        return name.title()
    zone_stats['Borough'] = zone_stats['Borough'].apply(clean_boro)
else:
    zone_stats['Borough'] = 'NYC'

cluster_means = zone_stats.groupby('CLUSTER')['VALUE_INDEX'].mean().sort_values()
cluster_labels = {
    cluster_means.index[0]: '1. The Gold Mines (Best Value)',
    cluster_means.index[1]: '2. Smart Suburbs (Good Value)',
    cluster_means.index[2]: '3. Premium Zones (Pricey)',
    cluster_means.index[3]: '4. Deal Breakers (Overpriced/Too Far)'
}
zone_stats['CLUSTER_NAME'] = zone_stats['CLUSTER'].map(cluster_labels)

zone_stats['RANK_IN_CLUSTER'] = zone_stats.groupby('CLUSTER_NAME')['VALUE_INDEX'].rank(method='min').astype(int)
zone_stats['TOTAL_IN_CLUSTER'] = zone_stats.groupby('CLUSTER_NAME')['CLUSTER_NAME'].transform('count')
zone_stats['RANK_TEXT'] = "อันดับ " + zone_stats['RANK_IN_CLUSTER'].astype(str) + " (จาก " + zone_stats['TOTAL_IN_CLUSTER'].astype(str) + " ย่าน)"

vmin = zone_stats['VALUE_INDEX'].quantile(0.05)
vmax = zone_stats['VALUE_INDEX'].quantile(0.95)

fig = px.choropleth_mapbox(
    zone_stats,
    geojson=nyc_geojson,
    locations=name_col,
    featureidkey=feature_key,
    color='VALUE_INDEX',
    color_continuous_scale="RdYlGn_r",
    range_color=[vmin, vmax],
    mapbox_style="carto-positron",
    zoom=9.5,
    center={"lat": 40.7128, "lon": -74.0060},
    opacity=0.7,
    hover_name=name_col,
    hover_data={
        'Borough': True,
        'CLUSTER_NAME': True,
        'RANK_TEXT': True,
        'PRICE': ':,.0f',
        'PRICE_PER_SQFT': ':,.0f',
        'DISTANCE_TO_CENTER': ':.3f',
        'VALUE_INDEX': ':.2f',
        name_col: False
    },
    title='💎 The Gold Mine Map: แผนที่ความคุ้มค่า พร้อมปักหมุดศูนย์กลางเศรษฐกิจ (Hubs)',
    width=1000, height=800
)
fig.update_traces(marker_line_color='white', marker_line_width=0.5)

hub_names = [
    "Midtown Manhattan", "Financial District",
    "Downtown Brooklyn", "Long Island City",
    "Flushing", "Jamaica", "The Hub / South Bronx", "St. George"
]
hub_lats = [h[0] for h in ECONOMIC_HUBS]
hub_lons = [h[1] for h in ECONOMIC_HUBS]

fig.add_trace(go.Scattermapbox(
    lat=hub_lats,
    lon=hub_lons,
    mode='markers',
    marker=go.scattermapbox.Marker(
        size=15,
        color='#FFD700',
        opacity=1.0
    ),
    text=hub_names,
    hoverinfo='text',
    name='Economic Hubs'
))

fig.update_layout(
    margin={"r":0,"t":50,"l":0,"b":0},
    coloraxis_colorbar=dict(title="ดัชนีความคุ้มค่า<br>(เขียวเข้ม=ดีสุด)"),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show(renderer="browser")

✅ 1. โหลดไฟล์สำเร็จ
2. กำลังคำนวณดัชนีความคุ้มค่า (ระบบ Multi-Hubs)...
3. กำลังดึงกรอบแผนที่...
✅ โหลดแผนที่ระดับย่าน (Neighborhoods) สำเร็จ!
📊 4. กำลังสรุปข้อมูล ดึงชื่อเขต และวาดแผนที่...


In [10]:
df_cleaned['CLUSTER'] = kmeans.fit_predict(X_scaled)
# ==========================================
# STEP 5: Export Dataset ใหม่
# ==========================================

output_path = "../data/NY_House_ValueMap.csv"

df_cleaned.to_csv(output_path, index=False)

print("✅ บันทึก Dataset ใหม่เรียบร้อย:", output_path)

✅ บันทึก Dataset ใหม่เรียบร้อย: ../data/NY_House_ValueMap.csv


In [3]:
cluster_labels = {
    cluster_means.index[0]: 'Gold Mine',
    cluster_means.index[1]: 'Smart Suburbs',
    cluster_means.index[2]: 'Premium Zone',
    cluster_means.index[3]: 'Overpriced'
}

df_cleaned['CLUSTER_NAME'] = df_cleaned['CLUSTER'].map(cluster_labels)

In [9]:
import joblib
# Dataset
df_cleaned.to_csv("../data/NY_House_ValueMap.csv", index=False)

# Model
joblib.dump(kmeans, "../app/ml_models/goldmine_model.pkl")

# Scaler
joblib.dump(scaler, "../app/ml_models/goldmine_scaler.pkl")

['../app/ml_models/goldmine_scaler.pkl']

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(n_neighbors=5)

model.fit(X_scaled)

import joblib

joblib.dump(model, "../app/ml_models/goal_minemap_model.pkl")
joblib.dump(scaler, "../app/ml_models/goal_minemap_scaler.pkl")

print("บันทึกโมเดลสำเร็จ")


บันทึกโมเดลสำเร็จ
